In [1]:
import os
import pickle
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time
import re
import pandas as pd
import sqlite3
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from datasets import Dataset
from itertools import chain
from accelerate import Accelerator

/home/manab/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Pretrained Model: Sentence-BERT
1. clean and encode data: SentenceTransformer
1. Compute Similarity cosine_similarity
1. Rank Job Descriptions
1. Enhancements
    - Section-Wise Comparison: If you extract sections like skills or experience from the job descriptions, calculate section-wise similarity to give more weight to specific areas.
    - Dynamic Weights: Assign weights dynamically based on the importance of each section.

In [2]:
"""
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = [
    cosine_similarity([my_resume_embedding], [jd_embedding])[0][0]
    for jd_embedding in job_description_embeddings
]

"""

'\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nsimilarity_scores = [\n    cosine_similarity([my_resume_embedding], [jd_embedding])[0][0]\n    for jd_embedding in job_description_embeddings\n]\n\n'

In [3]:
"""
# Pair job descriptions with their scores
job_matches = list(zip(job_descriptions, similarity_scores))

# Sort by similarity score
sorted_job_matches = sorted(job_matches, key=lambda x: x[1], reverse=True)

# Display top matches
for i, (jd, score) in enumerate(sorted_job_matches[:5]):  # Top 5 matches
    print(f"Rank {i + 1}: Score = {round(score * 100, 2)}%")
    print(f"Job Description: {jd}")
    print()

"""

'\n# Pair job descriptions with their scores\njob_matches = list(zip(job_descriptions, similarity_scores))\n\n# Sort by similarity score\nsorted_job_matches = sorted(job_matches, key=lambda x: x[1], reverse=True)\n\n# Display top matches\nfor i, (jd, score) in enumerate(sorted_job_matches[:5]):  # Top 5 matches\n    print(f"Rank {i + 1}: Score = {round(score * 100, 2)}%")\n    print(f"Job Description: {jd}")\n    print()\n\n'

In [4]:
"""
# Extract skills
job_skills = ["Python, SQL, Machine Learning"]  # Example
resume_skills = "Python, SQL, Docker"

# Skill similarity
skills_similarity = cosine_similarity(
    [model.encode(resume_skills)], [model.encode(job_skills)]
)[0][0]

"""

'\n# Extract skills\njob_skills = ["Python, SQL, Machine Learning"]  # Example\nresume_skills = "Python, SQL, Docker"\n\n# Skill similarity\nskills_similarity = cosine_similarity(\n    [model.encode(resume_skills)], [model.encode(job_skills)]\n)[0][0]\n\n'

In [5]:
conn = sqlite3.connect(R"job_table/jobs_details_old.db")
table_name= 'Linkedin'
df= pd.read_sql_query(f"SELECT * FROM {table_name} order by inserted_at desc;", conn)
# # df
# with open("table.html", "w") as file:
#     file.write(df.to_html(index=False))
df.info()
conn.close()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2071 entries, 0 to 2070
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   job_title                 2071 non-null   object
 1   job_location              2071 non-null   object
 2   job_posted                2071 non-null   object
 3   total_applicant           2071 non-null   object
 4   company_name              2071 non-null   object
 5   company_link              2071 non-null   object
 6   company_type              2071 non-null   object
 7   company_size              2071 non-null   object
 8   company_size_on_linkedin  2071 non-null   object
 9   job_description           2071 non-null   object
 10  job_linkedin_link         2071 non-null   object
 11  is_easy_apply             2071 non-null   object
 12  emails_found              2071 non-null   object
 13  inserted_at               2071 non-null   object
dtypes: object(14)
memory usa

In [6]:
job_desc_df= df["job_description"]
job_desc_df

0       About the job\nSkills:pyhon, Apache Airflow, S...
1       About the job\nClient Introduction:In this rol...
2       About the job\nTHIS IS A LONG TERM CONTRACT PO...
3       About the job\nGreetings from TCS!!!Walkin Int...
4       About the job\nSkills:Python programming, Data...
                              ...                        
2066    About the job\nSkills:Python (Programming Lang...
2067    About the job\nI am thrilled to share an excit...
2068    About the job\nStaff Software Developer (P4) (...
2069    About the job\nAbout:We are an industrial-AI s...
2070    About the job\nWe are seeking a skilled Python...
Name: job_description, Length: 2071, dtype: object

# Preprocessing

- All lower case
<!-- - Replace new line -->
- Remove special characters
- Remove URLs
- Encode

In [7]:
job_desc_df= job_desc_df.str.lower()    #.str.replace("\n", " ")

In [8]:
#remove links
job_desc_df= job_desc_df.apply(lambda text: re.sub(r'http\S+|www\S+', '', text.replace("about the job", "")))
job_desc_df[23]

"\nhello all,urgent requirement!!!designation: data science (python) trainerpositions – 3experience: 2 to 7 yearssalary: as per company standards.joining: immediatelyjob location: pune.job description : candidates must have the ability to take both theory and practical sessions.should have excellent knowledge on python data science technologies.should be able to teach machine learning, deep learning, artificial intelligence ,sql, djangoable to resolve students' queries.we expect good communication and presentation skills.able to explain complex subjects in a clear and interesting way.dynamic personality.should be flexible to conduct weekend batches as well as weekday batches.\nresponsibilities and duties : devise technical training programs according to organizational requirements.produce training schedules and classroom agenda.determine course content according to objectives.prepare training material (presentations, worksheets etc.)execute training sessionskeep and report data on comp

In [9]:
# make a serie of just one sentace as a row
job_desc_df= job_desc_df.apply(lambda text: [sentence.strip() for sentence in re.split(r'\n|\.', text) if sentence.strip()])
job_desc_df[23]

['hello all,urgent requirement!!!designation: data science (python) trainerpositions – 3experience: 2 to 7 yearssalary: as per company standards',
 'joining: immediatelyjob location: pune',
 'job description : candidates must have the ability to take both theory and practical sessions',
 'should have excellent knowledge on python data science technologies',
 "should be able to teach machine learning, deep learning, artificial intelligence ,sql, djangoable to resolve students' queries",
 'we expect good communication and presentation skills',
 'able to explain complex subjects in a clear and interesting way',
 'dynamic personality',
 'should be flexible to conduct weekend batches as well as weekday batches',
 'responsibilities and duties : devise technical training programs according to organizational requirements',
 'produce training schedules and classroom agenda',
 'determine course content according to objectives',
 'prepare training material (presentations, worksheets etc',
 ')exec

In [10]:
# make 1D series
job_desc_df= job_desc_df.explode().reset_index(drop=True).str.strip()
job_desc_df

0        skills:pyhon, apache airflow, sql, flask, djan...
1                                         python developer
2        job location- pune/bangalore/hyderabadjob desc...
3        expertise in python and its frameworksstrong u...
4                    experience working in sql environment
                               ...                        
40989    proven experience in etl processes and buildin...
40990    solid understanding of data cleaning and prepa...
40991        familiarity with relational databases and sql
40992    excellent problem-solving and communication sk...
40993    this role is perfect for candidates passionate...
Name: job_description, Length: 40994, dtype: object

In [11]:
# Remove special characters, keeping letters, numbers, spaces, and newlines
job_desc_df= job_desc_df.apply(lambda text: re.sub(r'[^A-Za-z0-9\s\n]', '', text))

job_desc_df

0        skillspyhon apache airflow sql flask django et...
1                                         python developer
2        job location punebangalorehyderabadjob descrip...
3        expertise in python and its frameworksstrong u...
4                    experience working in sql environment
                               ...                        
40989    proven experience in etl processes and buildin...
40990    solid understanding of data cleaning and prepa...
40991        familiarity with relational databases and sql
40992    excellent problemsolving and communication skills
40993    this role is perfect for candidates passionate...
Name: job_description, Length: 40994, dtype: object

In [25]:
job_desc_df= job_desc_df.replace("", None).dropna()
job_desc_df.info()

<class 'pandas.core.series.Series'>
Index: 40924 entries, 0 to 40993
Series name: job_description
Non-Null Count  Dtype 
--------------  ----- 
40924 non-null  object
dtypes: object(1)
memory usage: 639.4+ KB


### Categorize each line

will use that to remove unwanted lines from the actual data.

In [21]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
categories = ["Job Responsibilities", "Required Skills", "Company Information", "Other Details"]

Device set to use cuda:0


In [26]:
categry_data= job_desc_df.to_frame()
categry_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40924 entries, 0 to 40993
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   job_description  40924 non-null  object
dtypes: object(1)
memory usage: 639.4+ KB


In [27]:
categry_data["categore_lebels"]= categry_data["job_description"].apply(lambda text: classifier(text, categories)['labels'][0])
categry_data.head(4)

,job_description,categore_lebels
0,skillspyhon apache airflow sql flask django et...,Required Skills
1,python developer,Required Skills
2,job location punebangalorehyderabadjob descrip...,Job Responsibilities
3,expertise in python and its frameworksstrong u...,Required Skills


In [28]:
categry_data.to_csv("job_with_cat.csv")

In [29]:
df.to_csv("job_details.csv")